# Notebook 2 - Dataset Preparation

The classifier needs labelled examples - prompts tagged as either jailbreak (1) or benign (0).
This notebook loads the raw datasets, picks the right number of examples from each, and saves
three splits: train, val, and test.

## Step 1 - Imports and folder paths

Nothing special here - just `json` and `random` from the standard library, and `Path` so I don't have to hardcode OS-specific separators.

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

random.seed(42)  # makes sampling reproducible

ROOT_DIR       = Path.cwd().parent
DATA_RAW       = ROOT_DIR / 'data' / 'raw'
DATA_PROCESSED = ROOT_DIR / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## Step 2 - Helper functions

I need two tiny functions:

- **`load_jsonl`** - each dataset is stored as one JSON object per line. Python's `json.load` reads a single object, so I need a custom reader that handles the line-by-line format.
- **`write_jsonl`** - same idea in reverse: write each record as its own line so the output files stay consistent with the input format.

In [2]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def write_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'Saved {len(records)} records -> {path}')

## Step 3 - Load the raw datasets

Three sources, three roles:

| Dataset | Role | Why I need it |
|---|---|---|
| **JailBreakV-28K** | Jailbreak examples (label = 1) | 28K real attack prompts across 7 categories |
| **Stanford Alpaca** | Easy benign (label = 0) | Normal, clearly safe instruction prompts |
| **ToxicChat** (safe subset) | Hard benign (label = 0) | Sounds aggressive but is safe - stops the classifier being trigger-happy |

In [3]:
jb_raw     = load_jsonl(DATA_RAW / 'jailbreakv_28k.jsonl')
alpaca_raw = load_jsonl(DATA_RAW / 'alpaca_data.jsonl')
tc_raw     = load_jsonl(DATA_RAW / 'hard_benign_prompts.jsonl')

print(f'JailBreakV: {len(jb_raw)}')
print(f'Alpaca:     {len(alpaca_raw)}')
print(f'ToxicChat:  {len(tc_raw)}')

JailBreakV: 28000
Alpaca:     52002
ToxicChat:  4523


## Step 4 - Extract the text field and assign labels

The raw records have different key names depending on the dataset (e.g. `query`, `prompt`, `text`, `instruction`). I normalise them all into
a simple `{'text': ..., 'label': 0 or 1}` dict so everything is consistent before I do any sampling.

In [4]:
# jailbreak prompts - label 1
jailbreaks = []
for r in jb_raw:
    text = r.get('query') or r.get('prompt') or r.get('text', '')
    if text.strip():
        jailbreaks.append({'text': text, 'label': 1})

# easy benign - label 0
alpaca = []
for r in alpaca_raw:
    text = r.get('prompt', '')
    if text.strip():
        alpaca.append({'text': text, 'label': 0})

# hard benign (ToxicChat safe rows only) - label 0
hard_benign = []
for r in tc_raw:
    text = r.get('human_annotation') or r.get('input') or r.get('text', '')
    if text.strip():
        hard_benign.append({'text': text, 'label': 0})

print(f'Jailbreaks:  {len(jailbreaks)}')
print(f'Alpaca:      {len(alpaca)}')
print(f'Hard benign: {len(hard_benign)}')

Jailbreaks:  28000
Alpaca:      52002
Hard benign: 4523


## Step 5 - Sample and combine

I only need 1800 examples total:
- **900 jailbreaks** (label 1)
- **765 Alpaca** (85% of benign) - easy benign
- **135 ToxicChat** (15% of benign) - hard benign

The 15% hard-benign mix is deliberate. Too few and the classifier never learns to handle
tricky-but-safe prompts. Too many and it becomes overly cautious.

In [5]:
jb_sample  = random.sample(jailbreaks,  900)
alp_sample = random.sample(alpaca,      765)
hb_sample  = random.sample(hard_benign, 135)

# combine and shuffle so labels aren't grouped together
dataset = jb_sample + alp_sample + hb_sample
random.shuffle(dataset)

print(f'Total examples: {len(dataset)}')

Total examples: 1800


## Step 6 - Split into train / val / test

Standard 67 / 8 / 11 split (1200 / 150 / 200).
Because the dataset was shuffled in the previous step, I can just slice it.

In [6]:
train = dataset[:1200]
val   = dataset[1200:1350]
test  = dataset[1350:1550]

write_jsonl(train, DATA_PROCESSED / 'classifier_train.jsonl')
write_jsonl(val,   DATA_PROCESSED / 'classifier_val.jsonl')
write_jsonl(test,  DATA_PROCESSED / 'classifier_test.jsonl')

Saved 1200 records -> d:\National College Of Ireland\SEM-2\Dissertation\AdaptivePromptGuard\data\processed\classifier_train.jsonl
Saved 150 records -> d:\National College Of Ireland\SEM-2\Dissertation\AdaptivePromptGuard\data\processed\classifier_val.jsonl
Saved 200 records -> d:\National College Of Ireland\SEM-2\Dissertation\AdaptivePromptGuard\data\processed\classifier_test.jsonl


## Step 7 - Sanity check

Quick count to make sure both classes ended up in every split. If one split had zero jailbreaks the classifier would learn nothing useful from it.

In [7]:
for name, split in [('train', train), ('val', val), ('test', test)]:
    c = Counter(r['label'] for r in split)
    print(f'{name:6s}  benign={c[0]}  jailbreak={c[1]}')

train   benign=600  jailbreak=600
val     benign=74  jailbreak=76
test    benign=101  jailbreak=99
